In [1]:
import numpy as np
from osgeo import gdal
import os
import pandas as pd

In [2]:
def get_raster_file_list(path):
    File_list = [] #f for f in os.listdir(path) if os.isfile(mypath,f)
    for file in os.listdir(path):
        # "32628" is just to get here necessary ones
        if file.endswith(".tif") or file.endswith(".tiff"):
            if file not in File_list:
                File_list.append(os.path.join(path,file))
        else:
            pass
    return File_list

def get_unique_values_chunked(raster_path, chunk_size=1024, exclude_nodata=True):
    """
    Return unique pixel values of a raster band by reading in chunks.

    Parameters
    ----------
    raster_path: str, Path to raster file.
    chunk_size: int, optional Number of rows/cols per chunk to read at once (default: 1024).
    exclude_nodata: bool, optional Exclude NoData value from result (default: True).
    """
    ds = gdal.Open(raster_path)
    if ds is None:
        raise FileNotFoundError(f"Cannot open raster: {raster_path}")

    band = ds.GetRasterBand(1)
    nodata = band.GetNoDataValue()

    xsize = band.XSize
    ysize = band.YSize

    # Calculate how many chunks will be processed
    n_chunks_x = (xsize + chunk_size - 1) // chunk_size
    n_chunks_y = (ysize + chunk_size - 1) // chunk_size
    total_chunks = n_chunks_x * n_chunks_y

    unique_values = set()
    chunk_counter = 0

    print(f"Processing {total_chunks} chunks...")

    for y in range(0, ysize, chunk_size):
        rows = min(chunk_size, ysize - y)
        for x in range(0, xsize, chunk_size):
            cols = min(chunk_size, xsize - x)

            data = band.ReadAsArray(x, y, cols, rows)
            if data is None:
                continue

            if exclude_nodata and nodata is not None:
                data = data[data != nodata]

            unique_values.update(np.unique(data))

            # Update progress
            chunk_counter += 1
            if chunk_counter % 10 == 0 or chunk_counter == total_chunks:
                percent = (chunk_counter / total_chunks) * 100
                print(f"  Chunk {chunk_counter}/{total_chunks} ({percent:.1f}%)", end="\r", flush=True)

    ds = None  # close dataset
    return sorted(unique_values)


In [7]:
"""INPUTS"""
root_path = r"Z:\data\im-nca-colombia"
"Z:\data\im-nca-colombia\colombia_landcover_epsg_3116_2011.tif"
raster_list = get_raster_file_list(root_path)
raster_list[0:1]


['Z:\\data\\im-nca-colombia\\colombia_landcover_epsg_3116_2011.tif']

In [8]:
accumulative_df = []

for raster_path in raster_list[0:1]:
    unique_values = get_unique_values_chunked(raster_path, chunk_size=1024, exclude_nodata=True)
    raster_name = os.path.basename(raster_path).replace(".tif","")
    df = pd.DataFrame(sorted(unique_values), columns=[raster_name])
    accumulative_df.append(df)

cumulative_df = pd.concat(accumulative_df, axis=1)

Processing 4070 chunks...


In [9]:
cumulative_df.head()

,colombia_landcover_epsg_3116_2011
0,0.0
1,1.0
2,22.0
3,33.0
4,211.0


In [11]:
cumulative_df.to_csv(r"unique_values_new_2011.csv", index=False)